# Tutorial: benchmarking scRNA-seq batch integration with sctools

**Author:** Tahila Andrighetti  
**Last updated:** October 2026

> Code review, refactoring and documentation were developed with the assistance
> of [Claude Code](https://claude.com/claude-code) (Anthropic). All analytical
> choices were reviewed by the author.

This tutorial shows how to compare batch integration methods and preprocessing parameters with the `sctools.integration` module of the [sctools](https://github.com/TAndrighetti/SingleCell_Analysis_Tools) package.

The benchmark tests:

- 4 integration methods: `seurat` (CCA anchors), `harmony`, `scanorama` and `scvi`.
- 3 preprocessing parameters: HVG selection flavor, number of HVGs (`n_top_genes`) and number of PCs (`n_pcs`).

Each method and parameter set is scored with scIB metrics (Luecken et al., 2022). The result is a ranking of all combinations.

**Example data:** mouse proximal colon, 4 samples (2 control, 2 pectin diet), after quality control (see tutorial `01.QUALITY_CONTROL_tutorial.ipynb`).

**Requirements:**

- A conda environment with scanpy, scib, scvi-tools, harmonypy and scanorama.
- R with Seurat, used through rpy2, for the `seurat` method.
- An AnnData object after QC, with a raw count layer (integer counts) and a batch column in `.obs`.

**Steps:**

1. Load the data and create a normalized layer.
2. Define the parameter grid.
3. Run the benchmark.
4. Read the ranking.

In [1]:
# scanpy and pandas: AnnData objects and result tables
# sctools: preprocessing, integration and benchmarking functions
import scanpy as sc
import pandas as pd
import sctools

# Benchmark functions (sctools/integration/benchmarking.py)
from sctools.integration import BuildBenchmarkGrid, RunIntegrationBenchmark

/home/tahila/miniconda3/envs/scRNA_scib/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/tahila/miniconda3/envs/scRNA_scib/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/tahila/miniconda3/envs/scRNA_scib/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/tahila/miniconda3/envs/scRNA_scib/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/tahila/miniconda3/envs/scRNA_scib/lib/python3.10/site-packages/anndata/utils.py:

## 1. Load the data

Load the AnnData object after QC and create a normalized layer. The benchmark needs two layers:

| Layer | Content | Used by |
|---|---|---|
| `soupX_counts` | Raw counts (here, SoupX-corrected integer counts) | scVI, Seurat, and HVG selection with `flavor="seurat_v3"` |
| `QC_filtered_log1p` | Counts normalized to 10,000 per cell, then log1p | HVG selection (`seurat`, `cell_ranger`), PCA, and the Seurat "data" slot |

`NormalizeLog1pFromLayer` creates the log layer from a count layer. It does not change the count layer, but it overwrites `.X` with the normalized values. In this example, `QC_filtered` and `soupX_counts` hold the same counts.

`out_dir` is the output folder. `RunIntegrationBenchmark` creates a `tab_scib/` subfolder inside it.

In [2]:
import os

DATASET = 'scColonProx'

# Change these paths to your own data
dir = f"/media/tahila/DADOS/HD/2609_{DATASET}"
out_dir = f'{dir}/04.1.Integration_benchmark'   # RunIntegrationBenchmark creates "tab_scib" inside it
os.makedirs(out_dir, exist_ok=True)

# AnnData after QC (output of tutorial 01)
adata = sc.read_h5ad(f"{dir}/03.{DATASET}_QC_filtered.h5ad")
# Normalize to 10,000 counts per cell + log1p, saved in a new layer
sctools.NormalizeLog1pFromLayer(adata, input_layer='QC_filtered', output_layer='QC_filtered_log1p')

versao = f'04.1.{DATASET}'   # Optional prefix for all output files (passed as out_prefix)
adata

AnnData object with n_obs × n_vars = 31130 × 56748
    obs: 'sample', 'condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'doublet_score', 'predicted_doublet', 'is_doublet'
    var: 'gene_ids', 'feature_types', 'mt', 'ribo', 'hb'
    uns: 'qc_params', 'scrublet', 'log1p'
    layers: 'QC_filtered', 'counts', 'soupX_counts', 'QC_filtered_log1p'

## 2. Define the parameter grid

`BuildBenchmarkGrid` creates every combination of HVG flavor, `n_top_genes` and `n_pcs`. Each combination gets a `run_id`. Here we have 3 flavors x 3 HVG numbers x 3 PC numbers = 27 runs. Each run is tested with 4 methods, so 108 integrations in total.

Main settings:

- `batch_key`: the `.obs` column with the batch. Here each sample is one batch.
- `organism`: needed by the cell-cycle metric, which uses the cell-cycle genes of that species.
- `label_key`: `None` uses only label-free metrics. If you already have cell-type labels, give the `.obs` column name. Then label-based metrics are also computed (ASW, NMI, ARI, isolated labels, cLISI, kBET).

**Domain note:** `flavor="seurat_v3"` must run on raw counts, not on log data. sctools uses `counts_layer` for this flavor and `log_layer` for the other flavors.

In [3]:
# Benchmark settings
config = {
    "batch_key": "sample",
    "counts_layer": "soupX_counts",     # Raw counts: used by scVI, Seurat and HVG selection with flavor="seurat_v3"
    "log_layer": "QC_filtered_log1p",   # Normalized + log1p layer; must already exist (not computed here)
    "flavors": ("seurat", "seurat_v3","cell_ranger",),
    "n_top_genes_list": (1000,2000,3000,),   # Keep the trailing comma: (1000) is an int, (1000,) is a tuple
    "n_pcs_list": (20,35,50,),           # Same here
}

organism = "mouse"   # Required by the cell-cycle conservation metric
label_key = None     # None = label-free metrics; or the .obs column with cell-type labels

# One entry per combination: {run_id: {flavor, n_top_genes, n_pcs}}
combinations = BuildBenchmarkGrid(
    flavors=config["flavors"],
    n_top_genes_list=config["n_top_genes_list"],
    n_pcs_list=config["n_pcs_list"],
)
display(pd.DataFrame(combinations).T)   # Preview the grid before running

# Integration methods to test
methods = ("seurat", "harmony", "scanorama", "scvi")   

,flavor,n_top_genes,n_pcs
1,seurat,1000,20
2,seurat,1000,35
3,seurat,1000,50
4,seurat,2000,20
5,seurat,2000,35
6,seurat,2000,50
7,seurat,3000,20
8,seurat,3000,35
9,seurat,3000,50
10,seurat_v3,1000,20


## 3. Run the benchmark

For each run, `RunIntegrationBenchmark`:

1. Selects HVGs and runs PCA with the parameters of the run. This is the unintegrated baseline of that run.
2. Runs each integration method on the same HVGs.
3. Scores each method against the baseline of the same run, with scIB metrics.

Label-free metrics used here:

| Metric | Type | What it measures |
|---|---|---|
| `PCR_batch` | Batch removal | How much the variance explained by batch goes down, compared with the baseline |
| `iLISI` | Batch removal | How well the batches are mixed around each cell |
| `cell_cycle_conservation` | Bio conservation | How well the cell-cycle signal is kept after integration |
| `hvg_overlap` | Diagnostic only | Overlap of HVGs before and after integration. Computed only for `seurat` |

The full run can take many hours (27 runs x 4 methods; scVI is usually the slowest). This cell sends all the output to a log file, so the notebook stays small.

**Domain note:** `hvg_overlap` is not used in the ranking. Harmony, Scanorama and scVI only create a corrected embedding. Their expression matrix does not change, so `hvg_overlap` would always be 1.0 for them. Seurat is the only method here that corrects the expression matrix. Using this metric would give the other methods a perfect score for no real reason.

In [4]:
import contextlib
import logging

log_path = f"{out_dir}/{versao}.benchmark.log"

# Send all output (prints, warnings and sctools logs) to the log file.
# The sctools log handler was attached to the notebook output at import time,
# so redirect_stderr alone does not catch it. Swap it for a file handler.
sctools_logger = logging.getLogger("sctools")
old_handlers = sctools_logger.handlers[:]

with open(log_path, "w", buffering=1) as log_file, \
     contextlib.redirect_stdout(log_file), contextlib.redirect_stderr(log_file):
    file_handler = logging.StreamHandler(log_file)
    file_handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s:%(name)s:%(message)s"))
    sctools_logger.handlers = [file_handler]
    try:
        all_metrics_df, params_df, ranking = RunIntegrationBenchmark(
            adata=adata,
            combinations=combinations,
            out_dir=out_dir,
            batch_key=config["batch_key"],
            counts_layer=config["counts_layer"],
            log_layer=config["log_layer"],
            organism=organism,
            methods=methods,
            label_key=label_key,
            out_prefix=versao,
        )
    finally:
        # Restore the notebook handler, even if the run fails
        sctools_logger.handlers = old_handlers
print(f"Done. Log: {log_path}")


Done. Log: /media/tahila/DADOS/HD/2609_scColonProx/04.1.Integration_benchmark/04.1.scColonProx.benchmark.log


### Output files

All files are saved in `{out_dir}/tab_scib/`, and each name starts with `{out_prefix}.`. The `.csv` files are tab-separated.

| File | Content |
|---|---|
| `scib_metrics_run{run_id}.csv` | Raw metrics of one run, one column per method |
| `scib_metrics_all_runs.csv` | Raw metrics of all runs together |
| `scib_params_all_runs.csv` | Parameters (flavor, n_top_genes, n_pcs) of each method and run |
| `all_metrics_raw.tsv` | Raw metrics, one row per method and run |
| `scib_metrics_summary.tsv` | Scaled metrics and scores, sorted by `overall` |
| `scib_metrics_with_params.tsv` | Same as the summary, plus the parameters of each run |

The last three files are written by `SummarizeScibBenchmarkResults`, which `RunIntegrationBenchmark` calls at the end. `RunIntegrationBenchmark` also returns three tables: `all_metrics_df`, `params_df` and `ranking`.

## 4. Results

`SummarizeScibBenchmarkResults` reads the per-run files and builds the ranking. You can run it again later, without repeating the benchmark.

How the scores are computed:

1. Each metric is min-max scaled to [0, 1] across all method and run combinations. If a metric has the same value in all combinations, it keeps its raw value.
2. `batch_score` = mean of the scaled batch metrics (`PCR_batch`, `iLISI`).
3. `bio_score` = mean of the scaled bio metrics (here, only `cell_cycle_conservation`).
4. `overall` = 0.4 x `batch_score` + 0.6 x `bio_score`. These are the weights of the scIB benchmark (Luecken et al., 2022). You can change them with the `batch_weight` and `bio_weight` arguments.

The table is sorted by `overall`. The blue colors only help to read the table.

**Domain note:** min-max scaling makes the scores relative to the combinations tested here. An `overall` of 1.0 means "best in this benchmark", not "perfect integration". Scores from different benchmarks cannot be compared, unless they are scaled together (see the `sources` argument of `SummarizeScibBenchmarkResults`).

**Domain note:** without cell-type labels, `bio_score` depends only on the cell-cycle signal. A method can keep the cell cycle well and still mix different cell types. Before you choose a run, check the best ones on a UMAP with known marker genes. When you have cell-type labels, run the benchmark again with `label_key` to add label-based metrics.

**Next step:** use the chosen method and parameters with `RunIntegration` (`sctools/integration/workflow.py`).

**Reference:** Luecken MD, Büttner M, Chaichoompu K, et al. Benchmarking atlas-level data integration in single-cell genomics. *Nature Methods*. 2022;19:41-50.

In [5]:
# Rebuild the ranking from the files in tab_scib/ (no need to rerun the benchmark)
from sctools.integration import SummarizeScibBenchmarkResults

DATASET = 'scColonProx'

ranking_antib, _, _ = SummarizeScibBenchmarkResults(metrics_dir=f'{out_dir}/tab_scib', 
                                                    out_prefix=f'04.1.{DATASET}')

# Color the score columns (visual only)
score_cols = [c for c in ("PCR_batch", "iLISI", "cell_cycle_conservation","batch_score", "bio_score", "overall")
              if c in ranking_antib.columns]
ranking_antib.style.background_gradient(cmap="Blues", subset=score_cols)


,method,run_id,PCR_batch,iLISI,cell_cycle_conservation,batch_score,bio_score,overall,flavor,n_top_genes,n_pcs
0,seurat,19,0.731579,0.989312,1.000000,0.860445,1.000000,0.944178,cell_ranger,1000,20
1,seurat,21,0.751307,0.965694,1.000000,0.858500,1.000000,0.943400,cell_ranger,1000,50
2,seurat,20,0.739005,0.938741,1.000000,0.838873,1.000000,0.935549,cell_ranger,1000,35
3,harmony,12,0.932813,0.846852,0.956339,0.889833,0.956339,0.929736,seurat_v3,1000,50
4,seurat,13,0.782669,0.972374,0.911971,0.877522,0.911971,0.898191,seurat_v3,2000,20
5,seurat,12,0.817587,0.914680,0.919223,0.866133,0.919223,0.897987,seurat_v3,1000,50
6,seurat,15,0.831834,0.918811,0.911971,0.875323,0.911971,0.897312,seurat_v3,2000,50
7,seurat,14,0.819885,0.919711,0.911971,0.869798,0.911971,0.895102,seurat_v3,2000,35
8,seurat,11,0.816075,0.887817,0.919223,0.851946,0.919223,0.892312,seurat_v3,1000,35
9,seurat,10,0.746465,0.929128,0.919223,0.837797,0.919223,0.886653,seurat_v3,1000,20
